In [1]:
import os
import importlib
# os.environ["CUDA_VISIBLE_DEVICES"]="2,3"
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM, AutoModel
from datasets import load_dataset
import torch
#from sentence_transformers import SentenceTransformer, InputExample, losses
#from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator, SimilarityFunction
from torch.utils.data import DataLoader
from datasets import Dataset
import pandas as pd
from collections import defaultdict
import re
import numpy as np
from tqdm import tqdm

data_dir = '/raid/deallab/SF_RAG_Data/ASQA'
# data_dir = '../data'

device1 = 'cuda:2'
device2 = 'cuda:3'

gen_model_id = 'meta-llama/Meta-Llama-3.1-8B-Instruct'
# gen_model_id = 'mistralai/Mistral-7B-Instruct-v0.3'

split_token = '<|end_header_id|>'
end_token = '<|eot_id|>'

# split_token = '[/INST]'
# end_token = '</s>'

from evaluation import evaluate
import prompts
importlib.reload(prompts)

/home/dataconv/anaconda3/envs/sf_rag_djk/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<module 'prompts' from '/home/dataconv/deallab/djk/sf_rag/sf_rag/model/prompts.py'>

In [3]:
#load embeddings
embedd_test_path = f'{data_dir}/test/embedd_test.npy'
evidence_embeddings = np.load(embedd_test_path)
print(evidence_embeddings.shape)
evidence_embeddings = torch.from_numpy(evidence_embeddings).to(device1)

#load evidence
evidence_test_path = f'{data_dir}/test/evidence_test.csv'
evidence_df = pd.read_csv(evidence_test_path)

#load qa data
qa_df=pd.read_csv(f'{data_dir}/test/qa_test.csv') #data=df[['question','long_answers']] # questions=data['question'] #references = [row.to_dict() for i, row in df.iterrows() if i < len(questions)]
qa_df.head()

(21586, 4096)


,id,sample_id,question,follow_up_questions,long_answers,short_answers
0,c2687961-0957-45cb-bae0-42314e38f790,-7013890438520559398,Who has the highest goals in world football?,"[""Who has the highest goals in men's world int...","[""Ali Dael has the highest goals in men's worl...","[['Daei', 'Ali Daei'], ['Bican', 'Josef Bican'..."
1,26830122-8240-40a9-aaff-d9731d53b197,7089015503030534342,Who is the original artist of sound of silence?,['Who is the original artist of sound of silen...,[' The original artist of the song sound of si...,"[['Simon & Garfunkel', 'Paul Simon and Art Gar..."
2,268116a9-5ecb-4364-8da4-4a648f9d5b43,8793099883447006698,When was the first apple i phone made?,"['When was the first apple i phone released?',...",['The iPhone beta was created in 2004 to test ...,"[['June 29, 2007'], ['2004'], ['June 29, 2007...."
3,efb4810e-637b-4954-a776-3c2d05d1290c,-881464876144297194,Who played the weasley brothers in harry potter?,['Who played Bill weasley in Harry Potter and...,['Rupert Grint played Ron Weasley in all the H...,"[['Richard Fish'], ['Chris Rankin'], ['James P..."
4,99817eba-d32a-4c4d-9fe2-93a50ae1d367,1650309494326541834,How many state parks are there in virginia?,['How many state parks are there in virginia i...,['When the Virginia state park system was form...,"[['six'], ['38'], ['6'], ['38']]"


In [4]:
#load quantized model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_storage=torch.bfloat16,
)

# load model with tokenizer
model = AutoModel.from_pretrained(
    'nvidia/NV-Embed-v2', 
    trust_remote_code=True,
    quantization_config = bnb_config,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage =True,
)
model.eval()

Loading checkpoint shards: 100%|██████████| 4/4 [00:06<00:00,  1.53s/it]


NVEmbedModel(
  (latent_attention_model): LatentAttentionModel(
    (cross_attend_blocks): ModuleList(
      (0): PreNorm(
        (fn): Attention(
          (to_q): Linear4bit(in_features=4096, out_features=32768, bias=False)
          (to_kv): Linear4bit(in_features=4096, out_features=65536, bias=False)
          (to_out): Linear4bit(in_features=32768, out_features=4096, bias=False)
        )
        (norm): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
        (norm_context): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
      )
      (1): PreNorm(
        (fn): FeedForward(
          (net): Sequential(
            (0): Linear4bit(in_features=4096, out_features=32768, bias=True)
            (1): GEGLU()
            (2): Linear4bit(in_features=16384, out_features=4096, bias=True)
          )
        )
        (norm): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
      )
    )
  )
  (embedding_model): BidirectionalMistralModel(
    (embed_tokens): Embedding(

In [ ]:
#load tokenizer
tokenizer_gen = AutoTokenizer.from_pretrained(gen_model_id)
tokenizer_gen.pad_token = tokenizer_gen.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    # bnb_4bit_quant_type="nf4",
    # bnb_4bit_compute_dtype=torch.bfloat16,
    # bnb_4bit_use_double_quant=True,
    # bnb_4bit_quant_storage=torch.bfloat16,
)

model_gen = AutoModelForCausalLM.from_pretrained(
    gen_model_id,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    # device_map= 'auto'
)
model_gen.eval()

`low_cpu_mem_usage` was None, now default to True since model is quantized.
Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.05it/s]


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps

In [ ]:
# retrive docs from the document embeddings
def retrieve_documents(query):
    max_length = 1024
    
    #query prefix
    task_name_to_instruct = {"example": "Given a question, retrieve passages that answer the question",}
    query_prefix = "Instruct: "+task_name_to_instruct["example"]+"\nQuery: "
    
    query_embedding = model.encode([query],instruction=query_prefix, max_length=max_length).to(device1)

    # query_embedding = query_embedding.unsqueeze(0)
    #print(query_embedding)
    similarities = torch.nn.functional.cosine_similarity(query_embedding, evidence_embeddings)

    top_results = similarities.argsort(descending=True)[:10].cpu().detach().numpy()
    #print(top_results)
    res=[evidence_df.loc[idx, 'text'] for idx in top_results if idx < len(evidence_df)]
        
    return top_results, res

In [ ]:
def evaluate_docs(query, docs):
    # print(f"Query : {query}")
    # print("-"*100)
    outs = []
    for idx, doc in enumerate(docs):
        #print(f"Rank {idx} : {doc}")
        
        input= f'''
        Query: {query}
        Doc:  {doc}
        '''

        messages = [
            {"role":"user", 'content':prompts.PROMPT['eval_doc_instr']},
            {"role":"assistant", 'content':prompts.PROMPT['eval_doc_answ1']},
            {"role":"user", 'content':prompts.PROMPT['eval_doc_ex2']},
            {"role":"assistant", 'content':prompts.PROMPT['eval_doc_answ2']},
            {"role":"user", 'content':prompts.PROMPT['eval_doc_ex3']},
            {"role":"assistant", 'content':prompts.PROMPT['eval_doc_answ3']},
            {"role":"user", 'content':input}, 
        ]
        #apply tokenizter + generate eval
        inputs = tokenizer_gen.apply_chat_template(messages, return_tensors="pt", truncation=True).to(device1)
        
        attention_mask = (inputs != tokenizer_gen.pad_token_id).long().to(device1)
        
        outputs = model_gen.generate(inputs, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens=128)
        generated_text = tokenizer_gen.decode(outputs[0]).split(split_token)[-1].replace(end_token, '').strip('\n')
        
        if '#relevant' in generated_text:
            outs.append(doc)
    
    return outs

In [ ]:
def make_new_query(query,context):
    
    input= f'''
    Original Query: {query}
    Context information: {context}
    '''
    
    messages = [
        {"role":"user", 'content':prompts.PROMPT['refine_query_instr']},
        {"role":"assistant", 'content':prompts.PROMPT['refine_query_answ1']},
        {"role":"user", 'content':prompts.PROMPT['refine_query_ex2']},
        {"role":"assistant", 'content':prompts.PROMPT['refine_query_answ2']},
        {"role":"user", 'content':input},
    ]
    inputs = tokenizer_gen.apply_chat_template(messages, return_tensors="pt", truncation=True).to(device1)
    
    attention_mask = (inputs != tokenizer_gen.pad_token_id).long().to(device1)
    
    outputs = model_gen.generate(inputs, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens=512)
    generated_text = tokenizer_gen.decode(outputs[0]).split(split_token)[-1].replace(end_token, '').strip('\n')
    generated_text  = generated_text.strip('[]').split('\n')
    
    print(generated_text)
    return generated_text

In [ ]:
# def preprocessing(new_questions):
#     return list(((new_questions.split("['")[1]).split("']")[0]).split("',\n    '"))

In [ ]:
# preprocessing(make_new_query(query, rel_docs))

In [ ]:
def make_new_answer(query,context):
    
    context_str = '\n'.join(context)
    
    input= f'''
    Original Query: {query}
    Context information: {context_str}
    '''
    
    messages = [
        {"role":"user", 'content':prompts.PROMPT['new_answer_instr']},
        {"role":"assistant", 'content':prompts.PROMPT['new_answer_answ1']},
        {"role":"user", 'content':prompts.PROMPT['new_answer_ex2']},
        {"role":"assistant", 'content':prompts.PROMPT['new_answer_answ2']},
        {"role":"user", 'content':input},
    ]
    inputs = tokenizer_gen.apply_chat_template(messages, return_tensors="pt", truncation=True).to(device1)
    
    attention_mask = (inputs != tokenizer_gen.pad_token_id).long().to(device1)
    print(len(inputs[0]))
    outputs = model_gen.generate(inputs, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens=256)
    generated_text = tokenizer_gen.decode(outputs[0]).split(split_token)[-1].replace(end_token, '').strip('\n')
    
    print(f'New Answer: {generated_text}')
    return generated_text

In [ ]:
# import re

# def summarize_answers(question, answers):
#     input= f'''
#     Query: {query}
#     Context information: {answers}
#     '''
    
#     messages = [
#         {"role":"user", 'content':prompts.PROMPT['final_answer_instr']},
#         {"role":"assistant", 'content':prompts.PROMPT['final_answer_answ1']},
#         {"role":"user", 'content':{input}},
#     ]

#     #tokenizer prompt
#     input_ids = tokenizer_gen.apply_chat_template(messages, return_tensors="pt", truncation=True).to(device)

#     attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device)

#     out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
#     res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
#     candidate = [re.sub('\n|<\|eot_id\|>', '', res)]
#     return candidate

In [ ]:
# data=qa_df[['question','long_answers']]
# questions=data['question']

In [ ]:
# references = [row.to_dict() for i, row in qa_df.iterrows() if i < len(questions)]

In [ ]:
# references[0]

In [ ]:
def initial_answer(query, docs):
    prompt = """
    Context information is below.
    ---------------------
    {0}
    ---------------------
    Given the context information and not prior knowledge, answer the query.
    Query: {1}
    Answer:
    """.format('\n'.join(docs), query)
    input_ids = tokenizer_gen.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt').to(device2)

    attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device2)

    out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
    res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
    text = re.sub('\n|<\|eot_id\|>', '', res)
    return text

In [ ]:
def final_ans(query,answer, qa_pairs):
    # prompt = f"""
    # Context information is below.
    # ---------------------
    # {answers}
    # ---------------------
    # Given the context information and not prior knowledge, 
    # Answer questions that have multiple correct answers based on multiple interpretations, including multiple answers.
    # Query: {query}
    # Answer:
    # """
    
    # input_ids = tokenizer_gen.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt').to(device1)
    qa_sample = '''Follow-up Query{i}: {q}
    Context: {context}
    '''
    qa_string = '\n'.join([qa_sample.format(i=i, q=q, context=a) for i, (q, a) in enumerate(qa_pairs)])
    
    input= f'''
    Initial Query: {query}
    Context: {answer}
    {qa_string}
    '''
    print(f'Final Answ Input:{input}')
    messages = [
        {"role":"user", 'content':prompts.PROMPT['final_answer_instr']},
        {"role":"assistant", 'content':prompts.PROMPT['final_answer_answ1']},
        {"role":"user", 'content':input},
    ]

    #tokenizer prompt
    input_ids = tokenizer_gen.apply_chat_template(messages, return_tensors="pt", truncation=True).to(device2)

    attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device2)

    out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
    res = tokenizer_gen.decode(out[0]).split(split_token)[-1] 
    candidate = re.sub(end_token, '', res)
    #print(candidate)
    return candidate

In [ ]:
from evaluation import evaluate
from collections import defaultdict

scores_list=[]
stop_iteration=120
start=0
test_df=qa_df.iloc[start:start+stop_iteration]

new_answers_dic=defaultdict(list)

for idx, row in tqdm(test_df.iterrows(), total=min(len(test_df), stop_iteration)):
    # if idx == stop_iteration: break
    query = row['question']
    
    #retrieve relevant docs
    ids, docs = retrieve_documents(query)
    init_answer=initial_answer(query, docs)
    print(f'Initial Answer: {init_answer}')
    
    rel_docs = evaluate_docs(query, docs)
    rel_answer = make_new_answer(query, rel_docs)
    print(f'Relevant Answer: {rel_answer}')
    
    # generate new queries
    new_queries=make_new_query(query, rel_docs)
    
    #iterate over new docs
    qa_pairs = []
    for i, new_query in tqdm(enumerate(new_queries)):
        if i == 8: break #brak after x follow-up question 
        
        # retrieve relevant docs
        ids, new_docs=retrieve_documents(new_query)
        new_rel_docs=evaluate_docs(new_query, new_docs)
        if not new_rel_docs: continue
        new_answer = make_new_answer(new_query, new_rel_docs)
        qa_pairs.append((new_query, new_answer))

    # generate final answer
    add_answer=final_ans(query, rel_answer, qa_pairs)
    print(f'candidate: {init_answer+rel_answer+add_answer}')
    # print(references[i])
    print([row.to_dict()])
    scores=evaluate([init_answer+rel_answer+add_answer],[row.to_dict()])
    print(scores)
    scores_list.append(scores)
    scores_df=pd.DataFrame(scores_list)
    scores_df.to_csv('./results/self-refine-11-25_results.csv', index=False)
    
scores_df=pd.DataFrame(scores_list)
print(scores_df.mean())
scores_df.to_csv('./results/self-refine-11-25_results.csv', index=False)

  0%|          | 0/120 [00:00<?, ?it/s]/home/dataconv/.cache/huggingface/modules/transformers_modules/nvidia/NV-Embed-v2/7604d305b621f14095a1aa23d351674c2859553a/modeling_nvembed.py:349: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'input_ids': torch.tensor(batch_dict.get('input_ids').to(batch_dict.get('input_ids')).long()),
/home/dataconv/anaconda3/envs/sf_rag_djk/lib/python3.10/contextlib.py:103: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


Initial Answer: According to the provided information, Cristiano Ronaldo holds the record for the highest goals in men's international football with 133 goals, and Miroslav Klose holds the record for the highest goals in the World Cup with 16 goals.
5837
New Answer: Cristiano Ronaldo holds the record for the most international goals scored with 133 goals in 216 appearances for the Portugal national team. He is the highest overall men's international goalscorer in history.
Relevant Answer: Cristiano Ronaldo holds the record for the most international goals scored with 133 goals in 216 appearances for the Portugal national team. He is the highest overall men's international goalscorer in history.
["'### What is the total number of players who have scored 50 or more international goals?',", "'### What is the average number of goals scored by players who have achieved this milestone?',", "'### Which countries have the most players who have scored 50 or more international goals?',", "'### W

1688


New Answer: 81 male footballers have scored at least 50 goals with their national team at senior level. Cristiano Ronaldo holds the all-time record with 133 international goals. Brazil, Hungary, Iran, and Kuwait hold the record of having the most players to have scored 50 or more international goals with four each.
1207


New Answer: There is no definitive answer to the average number of goals scored by players who have achieved the milestone of 500 or more goals, as there are varying records and discrepancies among different sources and organisations. However, according to the RSSSF, the average number of goals scored by the 77 players who have achieved this milestone is not explicitly stated.
1687


...

candidate: Harlan PuckettHarlan Puckett was killed by Louise after she intervened in his attempted rape of Thelma.

Harlan Puckett was killed by Louise after she intervened in his attempted rape of Thelma, sparking a chain reaction of consequences for the two women that led to their decision to flee to Mexico and evade the law. This event was a traumatic experience that led to Louise's actions and ultimately resulted in the tragic deaths of Thelma and Louise. The attempted rape and subsequent killing of Harlan Puckett by Louise was a pivotal moment in the lives of the two women, leading to a series of events that changed their lives forever.||
[{'id': '7791c6c8-27e0-43c0-936d-eb6de7e9b7ca', 'sample_id': -3322598412088356524, 'question': 'Who killed the man in thelma and louise?', 'follow_up_questions': "['Which character killed the man in thelma and louise?', 'Which actor killed the man in thelma and louise?', 'Who is the character that kills Harlan in the film Thelma and Louise?', 'Wh

  6%|▌         | 7/120 [18:20<4:17:11, 136.56s/it]

Which character killed the man in thelma and louise?
{'score': 0.07176162302494049, 'start': 0, 'end': 20, 'answer': 'Harlan PuckettHarlan'}
{'score': 0.07176162302494049, 'start': 0, 'end': 20, 'answer': 'Harlan PuckettHarlan'}
['Louise Elizabeth Sawyer', 'Louise']
Which actor killed the man in thelma and louise?
{'score': 0.13633553683757782, 'start': 0, 'end': 20, 'answer': 'Harlan PuckettHarlan'}
{'score': 0.13633553683757782, 'start': 0, 'end': 20, 'answer': 'Harlan PuckettHarlan'}
['Susan Sarandon', 'Susan Abigail Sarandon']
Who is the character that kills Harlan in the film Thelma and Louise?
{'score': 0.0784633681178093, 'start': 0, 'end': 20, 'answer': 'Harlan PuckettHarlan'}
{'score': 0.0784633681178093, 'start': 0, 'end': 20, 'answer': 'Harlan PuckettHarlan'}
['Louise Elizabeth Sawyer', 'Louise']
Who is the actor of the character that killed a man in the film Thelma and Louise?
{'score': 0.7540242671966553, 'start': 0, 'end': 20, 'answer': 'Harlan PuckettHarlan'}
{'score': 0

  6%|▌         | 7/120 [18:23<4:56:54, 157.65s/it]


KeyboardInterrupt: 

In [ ]:
from evaluation import evaluate
scores=evaluate([init_answer+rel_answer+add_answer],[row.to_dict()])

{'id': 'f743f676-48f8-42c6-ab8e-e4cd0a0542ce', 'sample_id': -7013890438520559398, 'question': 'Who has the highest goals in world football?', 'follow_up_questions': '["Who has the highest goals in men\'s world international football?", "Who has the highest goals all-time in men\'s football?", "Who has the highest goals in women\'s world international football?"]', 'long_answers': '["Ali Dael has the highest goals in men\'s world international football with 109 goals. Josef Bican has the highest goals all-time in men\'s football and Christine Sinclair has the highest goals in women\'s world international football.", "The players with the highest all-time goals and highest men\'s and women\'s international football goals differ. The player with the highest all-time men\'s football goals is Josef Bican, who in 2020 was recognized by FIFA, the international governing body of football, as the record scorer with an estimated 805 goals. Christine Sinclair has the highest goals in women\'s int

KeyError: 'Disambig-F1'

In [ ]:
#  20%|██        | 1/5 [01:19<05:16, 79.10s/it]
# ['Based on the provided context information, there are multiple answers to the query "Who has the highest goals in world football?" depending on the interpretation.1. **Cristiano Ronaldo**: With 133 international goals, Cristiano Ronaldo holds the record for the highest number of goals scored in international football.2. **Cristiano Ronaldo (in European football)**: Ronaldo also holds the record for the highest number of goals scored in European football, with 85 international goals.3. **Cristiano Ronaldo (in European Championship)**: He is the first player to score 14 goals at the European Championships.4. **Cristiano Ronaldo (in UEFA Nations League)**: Ronaldo is the top scorer in the inaugural UEFA Nations League, with 5 goals.5. **Pelé**: He was the first player from South America to score at least 50 international goals and went on to score 77 international goals in 92 matches.6. **Mokhtar Dahari**: He broke the record for the highest international goalscorer, scoring 89 goals for Malaysia in 142 international appearances.7. **Imre Schlosser**: He was the first player to score 50 international goals and held the record for 26 years until Ferenc Puskás broke it.8. **Ferenc Puskás**: He broke the record for the highest international goalscorer, scoring 84 goals in his international career.9. **Vivian Woodward**: He was the fastest to achieve the feat of 50 international goals, scoring his 50th goal in his 32nd official international match.10. **Lionel Messi**: He became the third player to reach and pass the milestone of 100 international goals, as well as the first South American to achieve the feat.These are just a few examples of players who have achieved significant milestones in international football.']
# {'rougeLsum': 27.368421052631582, 'length': 264.0, 'str_em': 0.0, 'ovscore': 0.0}
#  40%|████      | 2/5 [02:18<03:22, 67.38s/it]
# ['The original artist of "The Sound of Silence" is Simon & Garfunkel, specifically Paul Simon, who wrote the song, and Art Garfunkel, who sang the melody.']
# {'rougeLsum': 41.463414634146346, 'length': 26.0, 'str_em': 66.66666666666666, 'ovscore': 52.57592264788534}
#  60%|██████    | 3/5 [03:09<02:00, 60.04s/it]
# ['The development of the first Apple iPhone began in 2005, and it was officially announced on January 9, 2007.']
# {'rougeLsum': 28.915662650602407, 'length': 19.0, 'str_em': 0.0, 'ovscore': 0.0}
#  80%|████████  | 4/5 [04:01<00:56, 56.98s/it]
# ['The Weasley brothers were portrayed by James and Oliver Phelps, who played Fred and George Weasley respectively.']
# {'rougeLsum': 19.607843137254903, 'length': 17.0, 'str_em': 16.666666666666664, 'ovscore': 18.07753815155468}
#  80%|████████  | 4/5 [04:12<01:03, 63.23s/it]

In [ ]:
scores_df=pd.DataFrame(scores_list)

In [ ]:
scores_df.mean()

# rougeLsum    29.178478
# length       93.666667
# str_em       30.555556
# ovscore      18.200875
# dtype: float64
